# Фінальний пайплайн: ансамбль моделей (Weighted Boxes Fusion)
WBF не відкидає перекриті бокси, як NMS, а усереднює координати й впевненості боксів різних моделей, що збігаються (IoU ≥ thr).
Набори моделей і ваги підбираються на val, найкращий застосовується до тесту.
Input: дані змагання + Output ноутбуків з `val_preds.pkl` / `test_preds.pkl` (RT-DETR, фінальний F1, YOLO, Faster R-CNN). Accelerator: None.

## 1. Середовище, предикти всіх моделей, оцінка

In [1]:
import importlib.util, subprocess, sys
for pkg, pip_name in [("pycocotools", "pycocotools"), ("ensemble_boxes", "ensemble-boxes")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import contextlib, io, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
from ensemble_boxes import weighted_boxes_fusion
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

MAX_DET, TOPK = 500, 200                # ≤500 у сабміті; на вхід WBF — top-200 кожної моделі (швидкість)
ROOT = next(p.parent.parent for p in Path("/kaggle/input").rglob("val/annotations_coco.json"))
RUNS = {p.parent.name: p.parent for p in sorted(Path("/kaggle/input").rglob("val_preds.pkl"))
        if (p.parent / "test_preds.pkl").exists()}
OUT = Path("/kaggle/working")
print("дані:", ROOT, "\nмоделі з предиктами:", list(RUNS))

with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(str(ROOT / "val" / "annotations_coco.json"))
KEY2ID, SIZE = {}, {}
for im in GT.dataset["images"]:
    k = (Path(im["file_name"]).parent.name, Path(im["file_name"]).stem)
    KEY2ID[k] = im["id"]; SIZE[k] = (im["width"], im["height"])
test_df = pd.read_csv(ROOT / "test_images.csv")
for fn, w, h in zip(test_df.file_name, test_df.width, test_df.height):
    SIZE[(Path(fn).parent.name, Path(fn).stem)] = (w, h)


def load(run, tag):
    raw = pickle.load(open(RUNS[run] / f"{tag}_preds.pkl", "rb"))
    return {k: (np.asarray(b, np.float32), np.asarray(s, np.float32), np.asarray(c, np.int16)) for k, (b, s, c) in raw.items()}


def evaluate(preds):
    dets = []
    for k, (b, s, c) in preds.items():
        for i in np.argsort(-s)[:MAX_DET]:
            x1, y1, x2, y2 = map(float, b[i])
            dets.append(dict(image_id=KEY2ID[k], category_id=int(c[i]) + 1, bbox=[x1, y1, x2 - x1, y2 - y1], score=float(s[i])))
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(GT, GT.loadRes(dets), "bbox")
        E.params.imgIds = [KEY2ID[k] for k in preds]
        E.params.maxDets = [1, 100, MAX_DET]
        E.evaluate(); E.accumulate(); E.summarize()
    p = E.eval["precision"][:, :, :, 0, -1]
    return dict(mAP=float(p[p > -1].mean()), AP50=float(E.stats[1]), AP_small=float(E.stats[3]))


VAL = {r: load(r, "val") for r in RUNS}
single = pd.DataFrame({r: evaluate(VAL[r]) for r in RUNS}).T.sort_values("mAP", ascending=False)
display(single.round(4))

дані: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026 
моделі з предиктами: ['A2_yolo11s_p2_1024', 'A3_yolo11m_1280', 'A4_yolo11s_p2_1280', 'R0_retina_1024', 'R1_retina_anch8_1024', 'R2_retina_anch8_ciou_1024', 'T0_frcnn_1024', 'Q0_rtdetr_l_1024', 'Q1_rtdetr_l_nq500_1024']


,mAP,AP50,AP_small
Q0_rtdetr_l_1024,0.2683,0.5135,0.1389
Q1_rtdetr_l_nq500_1024,0.2613,0.4997,0.1245
A3_yolo11m_1280,0.2281,0.4366,0.1284
T0_frcnn_1024,0.2148,0.4227,0.0934
A4_yolo11s_p2_1280,0.1952,0.3973,0.1082
A2_yolo11s_p2_1024,0.1846,0.3657,0.1078
R0_retina_1024,0.1516,0.2890,0.0549
R1_retina_anch8_1024,0.0618,0.1858,0.0311
R2_retina_anch8_ciou_1024,0.0600,0.1840,0.0324


## 2. WBF

In [2]:
def wbf(model_preds, weights, iou_thr=0.6, skip=0.001):
    """model_preds: список словників предиктів (однакові ключі кадрів). Координати нормалізуються до [0, 1] для WBF."""
    out = {}
    for k in model_preds[0]:
        W, H = SIZE[k]; scale = np.array([W, H, W, H], np.float32)
        bl, sl, ll = [], [], []
        for p in model_preds:
            b, s, c = p.get(k, (np.zeros((0, 4), np.float32), np.zeros(0, np.float32), np.zeros(0, np.int16)))
            o = np.argsort(-s)[:TOPK]
            bl.append(np.clip(b[o] / scale, 0, 1)); sl.append(s[o]); ll.append(c[o])
        b, s, c = weighted_boxes_fusion(bl, sl, ll, weights=weights, iou_thr=iou_thr, skip_box_thr=skip, conf_type="avg")
        out[k] = (np.asarray(b, np.float32) * scale, np.asarray(s, np.float32), np.asarray(c, np.int16))
    return out

## 3. Підбір набору моделей на val (IoU-поріг WBF 0.6)
Кандидати — найкраща модель і її комбінації з іншими (вага найкращої — 2, решти — 1).

In [3]:
best = single.index[0]
others = [r for r in single.index[1:4]]                      # до трьох наступних моделей
combos = [[best, o] for o in others] + ([[best] + others[:2]] if len(others) >= 2 else []) + ([[best] + others] if len(others) >= 3 else [])
rows = [dict(models=best, iou_thr=np.nan, **single.loc[best].to_dict())]
for models in combos:
    for thr in [0.6]:
        t0 = time.time()
        m = evaluate(wbf([VAL[r] for r in models], [2] + [1] * (len(models) - 1), iou_thr=thr))
        rows.append(dict(models=" + ".join(models), iou_thr=thr, **m))
        print(f"{' + '.join(models)} | thr {thr}: mAP {m['mAP']:.4f} ({time.time() - t0:.0f} с)")
grid = pd.DataFrame(rows).sort_values("mAP", ascending=False)
display(grid.round(4))
grid.to_csv(OUT / "wbf_grid.csv", index=False)

Q0_rtdetr_l_1024 + Q1_rtdetr_l_nq500_1024 | thr 0.6: mAP 0.2736 (120 с)


/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:67: UserWarning: Zero area box skipped: [0.6296875 1.        0.653125  1.       ].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:67: UserWarning: Zero area box skipped: [0.62578124 1.         0.6476562  1.        ].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:67: UserWarning: Zero area box skipped: [0.63554686 1.         0.6574218  1.        ].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:67: UserWarning: Zero area box skipped: [0.6253906  1.         0.64804685 1.        ].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:67: UserWarning: Ze

Q0_rtdetr_l_1024 + A3_yolo11m_1280 | thr 0.6: mAP 0.2767 (119 с)
Q0_rtdetr_l_1024 + T0_frcnn_1024 | thr 0.6: mAP 0.2799 (123 с)
Q0_rtdetr_l_1024 + Q1_rtdetr_l_nq500_1024 + A3_yolo11m_1280 | thr 0.6: mAP 0.2835 (150 с)
Q0_rtdetr_l_1024 + Q1_rtdetr_l_nq500_1024 + A3_yolo11m_1280 + T0_frcnn_1024 | thr 0.6: mAP 0.2944 (185 с)


,models,iou_thr,mAP,AP50,AP_small
5,Q0_rtdetr_l_1024 + Q1_rtdetr_l_nq500_1024 + A3...,0.6,0.2944,0.5546,0.1471
4,Q0_rtdetr_l_1024 + Q1_rtdetr_l_nq500_1024 + A3...,0.6,0.2835,0.5382,0.1420
3,Q0_rtdetr_l_1024 + T0_frcnn_1024,0.6,0.2799,0.5299,0.1399
2,Q0_rtdetr_l_1024 + A3_yolo11m_1280,0.6,0.2767,0.5211,0.1306
1,Q0_rtdetr_l_1024 + Q1_rtdetr_l_nq500_1024,0.6,0.2736,0.5226,0.1422
0,Q0_rtdetr_l_1024,NaN,0.2683,0.5135,0.1389


## 4. Найкращий ансамбль → тест → сабміт

In [4]:
top = grid.iloc[0]
print("найкраще на val:", top.models, "| iou_thr", top.iou_thr, "| mAP", round(top.mAP, 4))
if " + " in top.models:
    models = top.models.split(" + ")
    TEST = [load(r, "test") for r in models]
    fused = wbf(TEST, [2] + [1] * (len(models) - 1), iou_thr=top.iou_thr)
    rows = []
    for iid, fn in zip(test_df.image_id, test_df.file_name):
        b, s, c = fused.get((Path(fn).parent.name, Path(fn).stem), (np.zeros((0, 4)), np.zeros(0), np.zeros(0)))
        w, h = b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]
        keep = np.where((w >= 1) & (h >= 1))[0]
        o = keep[np.argsort(-s[keep])][:MAX_DET]
        rows.append((iid, " ".join(f"{int(c[i])} {s[i]:.4f} {b[i, 0]:.1f} {b[i, 1]:.1f} {w[i]:.1f} {h[i]:.1f}" for i in o)))
    pd.DataFrame(rows, columns=["image_id", "PredictionString"]).to_csv(OUT / "submission_wbf.csv", index=False)
    print("сабміт: submission_wbf.csv")
else:
    print("ансамбль не кращий за найкращу окрему модель — сабміт не потрібен")

найкраще на val: Q0_rtdetr_l_1024 + Q1_rtdetr_l_nq500_1024 + A3_yolo11m_1280 + T0_frcnn_1024 | iou_thr 0.6 | mAP 0.2944


/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:67: UserWarning: Zero area box skipped: [0.5875    1.        0.6070312 1.       ].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:67: UserWarning: Zero area box skipped: [0.5824219  1.         0.60429686 1.        ].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:67: UserWarning: Zero area box skipped: [0.00195312 0.         0.11015625 0.        ].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:67: UserWarning: Zero area box skipped: [0.         0.         0.08232422 0.        ].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:67: UserWarning: Ze

сабміт: submission_wbf.csv
